# Modelo com Filtro de Kalman

Este notebook aplica o Filtro de Kalman aos pares formados no Notebook 04.

A formação dos pares já foi feita anteriormente por:

1. pré-seleção por distância mínima;
2. teste de cointegração;
3. seleção dos 20 pares mais cointegrados por janela.

O objetivo aqui é estimar dinamicamente a relação entre os dois ativos de cada par, substituindo o beta fixo do OLS por um beta dinâmico estimado pelo Filtro de Kalman.

A relação estimada será:

log(ativo_1) = alpha_t + beta_t × log(ativo_2) + erro_t

O resultado será uma base com:

- alpha dinâmico;
- beta dinâmico;
- spread dinâmico;
- z-score dinâmico;
- sinais de trading.

### Importação das bibliotecas

In [1]:
import pandas as pd
import numpy as np

from pathlib import Path

pd.set_option("display.max_columns", 100)
pd.set_option("display.max_rows", 100)
pd.set_option("display.float_format", "{:,.6f}".format)

### Definição dos caminhos dos arquivos

Nesta etapa, definimos os arquivos que serão usados como entrada do modelo.

Usaremos duas bases:

1. a base de preços com setores, criada no Notebook 02;
2. a base de pares formados, criada no Notebook 04.

In [4]:
from pathlib import Path

arquivo_precos = Path("../dados_tratados/dados_economatica_B3_com_setores.parquet")

arquivo_pares = Path("../dados_tratados/pares_top20_cointegracao.parquet")

arquivo_saida = Path("../dados_tratados/modelo_kalman_sinais.parquet")

print("Arquivo de preços existe?", arquivo_precos.exists())
print("Arquivo de pares existe?", arquivo_pares.exists())

arquivo_saida.parent.mkdir(parents=True, exist_ok=True)

Arquivo de preços existe? True
Arquivo de pares existe? True


### Carregamento das bases

Nesta etapa, carregamos as bases necessárias para o modelo.

A base de preços será usada para extrair as séries de preço dos ativos.

A base de pares informa quais pares devem ser modelados em cada janela.

In [5]:
precos = pd.read_parquet(arquivo_precos)

pares = pd.read_parquet(arquivo_pares)

print("Base de preços:", precos.shape)
print("Base de pares:", pares.shape)

display(precos.head())
display(pares.head())

Base de preços: (1374399, 21)
Base de pares: (3460, 19)


,ticker,data,ativo,fechamento_ajustado,abertura_ajustada,minimo_ajustado,maximo_ajustado,medio_ajustado,q_negs,volume_financeiro,q_titulos,nome,classe,codigo_economatica,isin,id_papel,cnpj,situacao_cvm,setor,subsetor,segmento
0,ABYA3,2010-01-04,ABYA3<XBSP>,4.600000,4.610000,4.570000,4.620000,4.600000,463.000000,"3,991,454.000000","868,600.000000",Abyara,ON,ABYA3,NaN,ABYA3,07794351000160,CANCELADA,NaN,NaN,NaN
1,ABYA3,2010-01-05,ABYA3<XBSP>,4.580000,4.630000,4.570000,4.630000,4.600000,319.000000,"3,280,211.000000","713,700.000000",Abyara,ON,ABYA3,NaN,ABYA3,07794351000160,CANCELADA,NaN,NaN,NaN
2,ABYA3,2010-01-06,ABYA3<XBSP>,4.870000,4.570000,4.560000,4.920000,4.800000,"1,715.000000","18,347,711.000000","3,826,000.000000",Abyara,ON,ABYA3,NaN,ABYA3,07794351000160,CANCELADA,NaN,NaN,NaN
3,ABYA3,2010-01-07,ABYA3<XBSP>,5.170000,4.790000,4.740000,5.180000,5.030000,"2,655.000000","22,244,229.000000","4,420,100.000000",Abyara,ON,ABYA3,NaN,ABYA3,07794351000160,CANCELADA,NaN,NaN,NaN
4,ABYA3,2010-01-08,ABYA3<XBSP>,5.420000,5.180000,5.180000,5.450000,5.320000,"2,229.000000","21,789,270.000000","4,093,100.000000",Abyara,ON,ABYA3,NaN,ABYA3,07794351000160,CANCELADA,NaN,NaN,NaN


,data_fim_janela,ranking_cointegracao,ranking_distancia,ativo_1,ticker_1,setor_1,ativo_2,ticker_2,setor_2,mesmo_setor,combinacao_setorial,distancia,pvalor_coint,estatistica_coint,alpha,beta,media_spread,desvio_spread,obs_par
0,2012-01-31,1,49,BRTOYBACNOR4,TOYB3,Consumo cíclico,BRTOYBACNPR1,TOYB4,Consumo cíclico,True,Consumo cíclico | Consumo cíclico,2.018219,0.000001,-6.044942,0.191186,0.922945,-0.000000,0.118618,504
1,2012-01-31,2,312,BRBISAACNOR8,BISA3,-,BRRSIDACNOR8,RSID3,Consumo cíclico,False,- | Consumo cíclico,4.067568,0.000011,-5.598128,-3.009971,0.860943,-0.000000,0.041880,504
2,2012-01-31,3,378,BRBRMLACNOR9,BRML3,Financeiro,BRHBORACNOR3,HBOR3,Consumo cíclico,False,Financeiro | Consumo cíclico,4.321186,0.000049,-5.277648,0.069584,0.838683,0.000000,0.050878,504
3,2012-01-31,4,288,BRABCBACNPR4,ABCB4,Financeiro,BRITSAACNPR7,ITSA4,Financeiro,True,Financeiro | Financeiro,3.975838,0.000064,-5.218836,0.576766,1.343810,-0.000000,0.056568,504
4,2012-01-31,5,24,BRFLRYACNOR5,FLRY3,Saúde,BRWHRLACNPR2,WHRL4,Consumo cíclico,False,Saúde | Consumo cíclico,1.731670,0.000066,-5.210058,1.789616,1.063782,0.000000,0.047264,466


### Validação das colunas necessárias

Antes de aplicar o modelo, verificamos se as bases possuem as colunas necessárias.

Na base de preços, precisamos de:

- data;
- id_papel;
- fechamento_ajustado.

Na base de pares, precisamos de:

- data_fim_janela;
- ativo_1;
- ativo_2;
- ranking_cointegracao;
- pvalor_coint;
- distancia.

Também verificamos se a base possui alpha e beta estimados no período de formação. Esses valores serão usados como ponto inicial do Filtro de Kalman.

In [6]:
colunas_precos = [
    "data",
    "id_papel",
    "fechamento_ajustado"
]

colunas_pares = [
    "data_fim_janela",
    "ativo_1",
    "ativo_2",
    "ranking_cointegracao",
    "pvalor_coint",
    "distancia"
]

faltantes_precos = [
    coluna for coluna in colunas_precos
    if coluna not in precos.columns
]

faltantes_pares = [
    coluna for coluna in colunas_pares
    if coluna not in pares.columns
]

if faltantes_precos:
    raise ValueError(f"Colunas faltantes na base de preços: {faltantes_precos}")

if faltantes_pares:
    raise ValueError(f"Colunas faltantes na base de pares: {faltantes_pares}")

print("Todas as colunas obrigatórias estão presentes.")

Todas as colunas obrigatórias estão presentes.


### Padronização das datas e preparação da matriz de preços

Nesta etapa, garantimos que as datas estejam no formato correto e montamos a matriz de preços.

A matriz terá:

- linhas como datas;
- colunas como ativos identificados por `id_papel`;
- valores como preço de fechamento ajustado.

Essa estrutura facilita a extração das séries dos dois ativos de cada par.

In [7]:
precos = precos.copy()
pares = pares.copy()

precos["data"] = pd.to_datetime(precos["data"], errors="coerce")
pares["data_fim_janela"] = pd.to_datetime(pares["data_fim_janela"], errors="coerce")

precos = precos.dropna(subset=["data", "id_papel", "fechamento_ajustado"])

precos = precos[precos["fechamento_ajustado"] > 0].copy()

matriz_precos = (
    precos
    .pivot_table(
        index="data",
        columns="id_papel",
        values="fechamento_ajustado",
        aggfunc="last"
    )
    .sort_index()
)

matriz_log = np.log(matriz_precos)

print("Matriz de preços:", matriz_precos.shape)

display(matriz_precos.head())

Matriz de preços: (4053, 775)


id_papel,ABYA3,ACGU3,AEDU11,AGEI3,AGIN3,ALLL11,ALLL4,AVIL3,BNCA3,BRAALRACNOR6,BRABCBACNPR4,BRABEVACNOR1,BRABRECDAM15,BRADHMACNOR9,BRAEDUACNOR9,BRAELPACNOR2,BRAERIACNOR4,BRAESBACNOR7,BRAFLTACNOR1,BRAFLUACNOR9,BRAFLUACNPA2,BRAGENBDR001,BRAGROACNOR7,BRAGXYACNOR4,BRAHEBACNOR0,BRAHEBACNPA3,BRAHEBACNPB1,BRALLDACNOR3,BRALLLACNOR6,BRALOSACNOR5,BRALPAACNOR0,BRALPAACNPR7,BRALPKACNOR9,BRALSCACNOR0,BRALUPACNOR8,BRALUPACNPR5,BRALUPCDAM15,BRAMARACNOR4,BRAMBPACNOR6,BRAMBVACNPR1,BRAMERACNOR6,BRAMILACNOR0,BRAMOBACNOR9,BRAMPIACNOR1,BRANDGACNOR9,BRANIMACNOR6,BRAPERACNOR9,BRAPTIACNPR3,BRARMLACNOR1,BRARNDACNOR6,...,BRVIGRACNOR5,BRVINEACNOR9,BRVINEACNPA2,BRVINEACNPB0,BRVITTACNOR4,BRVIVAACNOR0,BRVIVRACNOR4,BRVIVTACNOR0,BRVIVTACNPR7,BRVLIDACNOR5,BRVSTEACNOR5,BRVTRUACNOR3,BRVULCACNOR2,BRVVARACNPR8,BRVVARCDAM10,BRVVEOACNOR0,BRWDCNACNOR2,BRWEGEACNOR0,BRWESTACNOR3,BRWHRLACNOR5,BRWHRLACNPR2,BRWISAACNOR4,BRWISAACNPR1,BRWIZCACNOR5,BRWLMMACNOR6,BRWLMMACNPR3,BRWMBYACNOR2,BRYDUQACNOR3,BRZAMPACNOR5,CYRE4,DXTG4,ELPL5,ELUM4,GVTT3,KSSA3,LEVE4,MEDI3,PMAM4,PNOR5,PNOR6,SEBB11,SZPQ4,TCSL4,TEND3-OLD,TRFO4,TROR4,VAGV4,VIVO3,VIVO4,VPSC4
data,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,
2010-01-04,4.600000,5.780000,25.096545,NaN,5.000000,17.038713,12.939803,0.635970,67.257849,NaN,3.828227,3.181294,NaN,NaN,NaN,31.287502,NaN,NaN,NaN,NaN,NaN,2.790000,4.531978,NaN,NaN,NaN,NaN,NaN,30.946646,NaN,NaN,2.086913,NaN,NaN,NaN,NaN,NaN,38.729817,NaN,6.227893,"3,863.522415",13.852121,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,"10,547.946999",5.547112,18.243788,7.582252,101.513501,NaN,13.273234,NaN,NaN,NaN,NaN,1.885304,NaN,NaN,0.834826,0.710000,0.770000,NaN,NaN,3.131678,NaN,5.076226,NaN,NaN,1.733255,NaN,NaN,55.950000,4.970000,23.458919,17.390000,6.600000,3.400000,2.921688,NaN,NaN,4.909862,65.179448,4.440000,1.550000,1.020000,47.602386,47.259290,0.009990
2010-01-05,4.580000,5.750000,24.776589,NaN,4.930000,18.207938,13.739173,0.626186,66.171452,NaN,3.930519,3.200008,NaN,NaN,NaN,32.628395,NaN,NaN,NaN,NaN,NaN,2.830000,NaN,NaN,NaN,NaN,NaN,NaN,31.190321,NaN,NaN,2.094172,NaN,NaN,NaN,NaN,NaN,40.238771,NaN,6.201943,"3,771.316225",13.635988,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,"10,487.151915",5.526557,18.117095,7.574395,99.662191,NaN,13.432513,NaN,NaN,NaN,NaN,1.881190,NaN,NaN,0.827711,NaN,NaN,NaN,NaN,3.171320,NaN,5.137032,NaN,NaN,1.733255,NaN,NaN,55.650000,4.970000,23.468269,17.400000,6.780000,3.440000,3.025361,15.992495,7.934857,4.975956,64.593302,NaN,NaN,NaN,NaN,48.221979,NaN
2010-01-06,4.870000,5.650000,25.196531,NaN,5.300000,17.558369,13.988976,0.635970,66.862796,NaN,3.995615,3.232652,NaN,NaN,NaN,32.658192,NaN,NaN,NaN,NaN,NaN,2.830000,NaN,NaN,NaN,NaN,NaN,NaN,32.408693,NaN,NaN,2.088728,NaN,NaN,NaN,NaN,NaN,40.205238,NaN,6.228945,"3,812.850545",13.950363,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,"11,155.897835",5.432588,18.281796,7.601895,99.045088,NaN,13.273234,NaN,NaN,NaN,NaN,1.891475,NaN,NaN,0.815852,NaN,NaN,NaN,NaN,3.250603,NaN,5.220902,NaN,NaN,1.733255,NaN,NaN,55.700000,5.330000,23.608518,17.340000,6.700000,3.580000,3.157308,16.097478,8.153066,5.070377,64.593302,4.440000,NaN,NaN,48.043149,47.749386,NaN
2010-01-07,5.170000,5.570000,25.436498,NaN,5.590000,17.318527,14.138858,0.635970,67.010941,NaN,3.936719,3.247831,NaN,NaN,NaN,32.658192,NaN,NaN,NaN,NaN,NaN,2.800000,4.660828,NaN,NaN,NaN,NaN,NaN,32.311223,NaN,2.218085,2.110504,NaN,NaN,NaN,NaN,NaN,41.177675,NaN,6.210710,"3,856.046237",14.441573,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,"11,247.090460",5.432588,18.188888,7.621538,98.736536,NaN,13.644885,NaN,NaN,NaN,NaN,1.923360,NaN,NaN,0.813481,NaN,NaN,NaN,NaN,3.262495,NaN,5.137032,NaN,NaN,1.761825,26.698821,NaN,55.500000,5.630000,23.608518,17.400000,6.740000,3.700000,3.675672,15.397588,8.232414,4.862652,64.241614,NaN,NaN,NaN,46.280097,46.559152,NaN
2010-01-08,5.420000,5.480000,27.196256,NaN,5.810000,17.488415,14.388661,0.635970,67.455376,NaN,3.905721,3.241593,NaN,NaN

### Parâmetros do Filtro de Kalman e dos sinais

Nesta etapa, definimos os parâmetros do modelo.

O parâmetro "KALMAN_DELTA" controla a velocidade de adaptação do alpha e do beta.

Os limites de z-score definem quando o modelo gera sinais:

- z-score maior ou igual a 2 indica que o ativo 1 está caro em relação ao ativo 2;
- z-score menor ou igual a -2 indica que o ativo 1 está barato em relação ao ativo 2.

In [8]:
KALMAN_DELTA = 1e-5

KALMAN_R_PADRAO = 1e-3

Z_ENTRADA = 2.0

MIN_OBS_TESTE = 5

### Identificação das colunas de alpha e beta da formação

O Notebook 04 pode ter salvo os parâmetros da regressão da formação com nomes diferentes.

Nesta etapa, identificamos automaticamente quais colunas contêm alpha, beta, média do spread e desvio do spread.

Esses parâmetros serão usados apenas como ponto inicial do Filtro de Kalman.

In [9]:
coluna_alpha = "alpha_ols_formacao" if "alpha_ols_formacao" in pares.columns else "alpha"

coluna_beta = "beta_ols_formacao" if "beta_ols_formacao" in pares.columns else "beta"

coluna_media_spread = (
    "media_spread_ols_formacao"
    if "media_spread_ols_formacao" in pares.columns
    else "media_spread"
)

coluna_desvio_spread = (
    "desvio_spread_ols_formacao"
    if "desvio_spread_ols_formacao" in pares.columns
    else "desvio_spread"
)

colunas_parametros = [
    coluna_alpha,
    coluna_beta,
    coluna_media_spread,
    coluna_desvio_spread
]

faltantes_parametros = [
    coluna for coluna in colunas_parametros
    if coluna not in pares.columns
]

if faltantes_parametros:
    raise ValueError(f"Colunas de parâmetros faltantes na base de pares: {faltantes_parametros}")

print("Coluna alpha usada:", coluna_alpha)
print("Coluna beta usada:", coluna_beta)
print("Coluna média do spread usada:", coluna_media_spread)
print("Coluna desvio do spread usada:", coluna_desvio_spread)

Coluna alpha usada: alpha
Coluna beta usada: beta
Coluna média do spread usada: media_spread
Coluna desvio do spread usada: desvio_spread


### Definição do período de teste de cada janela

Cada par foi formado em uma data chamada "data_fim_janela".

Para evitar look-ahead bias, o modelo não deve ser aplicado na mesma janela usada para formar o par.

Por isso, cada par será acompanhado apenas no período posterior à formação, até a próxima data de formação.

Exemplo:

- par formado em 31/01/2018;
- período de teste: depois de 31/01/2018 até 28/02/2018.

In [10]:
datas_janelas = sorted(pares["data_fim_janela"].dropna().unique())

proxima_janela = {
    datas_janelas[i]: datas_janelas[i + 1]
    for i in range(len(datas_janelas) - 1)
}

print("Quantidade de janelas:", len(datas_janelas))
print("Quantidade de janelas com período de teste:", len(proxima_janela))
print("Primeira janela:", datas_janelas[0])
print("Última janela:", datas_janelas[-1])

Quantidade de janelas: 173
Quantidade de janelas com período de teste: 172
Primeira janela: 2012-01-31 00:00:00
Última janela: 2026-05-31 00:00:00


### Função do Filtro de Kalman

Nesta etapa, criamos a função que aplica o Filtro de Kalman.

O modelo estima dinamicamente:

- alpha_t;
- beta_t.

A equação observada é:

log(ativo_1) = alpha_t + beta_t × log(ativo_2) + erro_t

O erro dessa equação é o spread dinâmico.

O z-score é calculado dividindo o erro pela incerteza estimada pelo próprio Kalman.

In [11]:
def aplicar_kalman(
    log_ativo_1,
    log_ativo_2,
    alpha_inicial,
    beta_inicial,
    variancia_erro=None,
    delta=KALMAN_DELTA
):
    y = log_ativo_1.values
    
    x = log_ativo_2.values
    
    datas = log_ativo_1.index
    
    theta = np.array([alpha_inicial, beta_inicial], dtype=float)
    
    P = np.eye(2)
    
    Q = delta / (1 - delta) * np.eye(2)
    
    if variancia_erro is None or variancia_erro <= 0:
        R = KALMAN_R_PADRAO
    else:
        R = variancia_erro
    
    resultados = []
    
    for i in range(len(y)):
        F = np.array([1.0, x[i]])
        
        theta_previsto = theta.copy()
        
        P_previsto = P + Q
        
        y_estimado = F @ theta_previsto
        
        erro = y[i] - y_estimado
        
        S = F @ P_previsto @ F.T + R
        
        K = (P_previsto @ F.T) / S
        
        theta = theta_previsto + K * erro
        
        P = P_previsto - np.outer(K, F) @ P_previsto
        
        zscore = erro / np.sqrt(S)
        
        resultados.append({
            "data": datas[i],
            "alpha_kalman": theta[0],
            "beta_kalman": theta[1],
            "spread_kalman": erro,
            "variancia_inovacao": S,
            "zscore_kalman": zscore
        })
    
    resultado = pd.DataFrame(resultados)
    
    return resultado

### Função para aplicar Kalman em um par

Nesta etapa, criamos uma função que aplica o Filtro de Kalman em um único par.

A função recebe uma linha da base de pares, identifica os ativos, define o período de teste e aplica o Kalman apenas depois da data de formação do par.

In [12]:
def aplicar_kalman_em_par(linha_par):
    data_formacao = linha_par["data_fim_janela"]
    
    data_fim_teste = proxima_janela.get(data_formacao)
    
    if data_fim_teste is None:
        return pd.DataFrame()
    
    ativo_1 = linha_par["ativo_1"]
    
    ativo_2 = linha_par["ativo_2"]
    
    if ativo_1 not in matriz_log.columns or ativo_2 not in matriz_log.columns:
        return pd.DataFrame()
    
    dados_teste = matriz_log.loc[
        (matriz_log.index > data_formacao) &
        (matriz_log.index <= data_fim_teste),
        [ativo_1, ativo_2]
    ].dropna()
    
    if len(dados_teste) < MIN_OBS_TESTE:
        return pd.DataFrame()
    
    alpha_inicial = linha_par[coluna_alpha]
    
    beta_inicial = linha_par[coluna_beta]
    
    desvio_spread = linha_par[coluna_desvio_spread]
    
    variancia_erro = desvio_spread ** 2
    
    resultado = aplicar_kalman(
        log_ativo_1=dados_teste[ativo_1],
        log_ativo_2=dados_teste[ativo_2],
        alpha_inicial=alpha_inicial,
        beta_inicial=beta_inicial,
        variancia_erro=variancia_erro
    )
    
    resultado["data_fim_janela"] = data_formacao
    resultado["data_fim_teste"] = data_fim_teste
    
    resultado["ativo_1"] = ativo_1
    resultado["ativo_2"] = ativo_2
    
    resultado["ranking_cointegracao"] = linha_par.get("ranking_cointegracao", np.nan)
    resultado["ranking_distancia"] = linha_par.get("ranking_distancia", np.nan)
    
    resultado["pvalor_coint"] = linha_par.get("pvalor_coint", np.nan)
    resultado["distancia"] = linha_par.get("distancia", np.nan)
    
    resultado["ticker_1"] = linha_par.get("ticker_1", np.nan)
    resultado["ticker_2"] = linha_par.get("ticker_2", np.nan)
    
    resultado["setor_1"] = linha_par.get("setor_1", np.nan)
    resultado["setor_2"] = linha_par.get("setor_2", np.nan)
    resultado["mesmo_setor"] = linha_par.get("mesmo_setor", np.nan)
    resultado["combinacao_setorial"] = linha_par.get("combinacao_setorial", np.nan)
    
    return resultado

### Aplicação do Kalman em todos os pares

Nesta etapa, aplicamos o Filtro de Kalman a todos os pares formados no Notebook 04.

Cada par é modelado apenas no período posterior à sua formação, evitando uso de informação futura.

In [13]:
resultados_kalman = []

for i, linha in pares.iterrows():
    resultado_par = aplicar_kalman_em_par(linha)
    
    if not resultado_par.empty:
        resultados_kalman.append(resultado_par)
    
    if (i + 1) % 100 == 0:
        print(f"Pares processados: {i + 1} de {len(pares)}")

if resultados_kalman:
    modelo_kalman = pd.concat(resultados_kalman, ignore_index=True)
else:
    modelo_kalman = pd.DataFrame()

print("Base do modelo Kalman:", modelo_kalman.shape)

display(modelo_kalman.head())

Pares processados: 100 de 3460
Pares processados: 200 de 3460
Pares processados: 300 de 3460
Pares processados: 400 de 3460
Pares processados: 500 de 3460
Pares processados: 600 de 3460
Pares processados: 700 de 3460
Pares processados: 800 de 3460
Pares processados: 900 de 3460
Pares processados: 1000 de 3460
Pares processados: 1100 de 3460
Pares processados: 1200 de 3460
Pares processados: 1300 de 3460
Pares processados: 1400 de 3460
Pares processados: 1500 de 3460
Pares processados: 1600 de 3460
Pares processados: 1700 de 3460
Pares processados: 1800 de 3460
Pares processados: 1900 de 3460
Pares processados: 2000 de 3460
Pares processados: 2100 de 3460
Pares processados: 2200 de 3460
Pares processados: 2300 de 3460
Pares processados: 2400 de 3460
Pares processados: 2500 de 3460
Pares processados: 2600 de 3460
Pares processados: 2700 de 3460
Pares processados: 2800 de 3460
Pares processados: 2900 de 3460
Pares processados: 3000 de 3460
Pares processados: 3100 de 3460
Pares processados

,data,alpha_kalman,beta_kalman,spread_kalman,variancia_inovacao,zscore_kalman,data_fim_janela,data_fim_teste,ativo_1,ativo_2,ranking_cointegracao,ranking_distancia,pvalor_coint,distancia,ticker_1,ticker_2,setor_1,setor_2,mesmo_setor,combinacao_setorial
0,2012-02-01,0.192486,0.927505,0.017306,13.309462,0.004744,2012-01-31,2012-02-29,BRTOYBACNOR4,BRTOYBACNPR1,1,49,0.000001,2.018219,TOYB3,TOYB4,Consumo cíclico,Consumo cíclico,True,Consumo cíclico | Consumo cíclico
1,2012-02-02,0.160653,0.934558,-0.016159,0.030383,-0.092702,2012-01-31,2012-02-29,BRTOYBACNOR4,BRTOYBACNPR1,1,49,0.000001,2.018219,TOYB3,TOYB4,Consumo cíclico,Consumo cíclico,True,Consumo cíclico | Consumo cíclico
2,2012-02-03,0.152677,0.937595,0.007120,0.022523,0.047443,2012-01-31,2012-02-29,BRTOYBACNOR4,BRTOYBACNPR1,1,49,0.000001,2.018219,TOYB3,TOYB4,Consumo cíclico,Consumo cíclico,True,Consumo cíclico | Consumo cíclico
3,2012-02-06,0.522811,0.844369,0.213666,0.020093,1.507353,2012-01-31,2012-02-29,BRTOYBACNOR4,BRTOYBACNPR1,1,49,0.000001,2.018219,TOYB3,TOYB4,Consumo cíclico,Consumo cíclico,True,Consumo cíclico | Consumo cíclico
4,2012-02-07,0.425407,0.868761,-0.073522,0.018405,-0.541934,2012-01-31,2012-02-29,BRTOYBACNOR4,BRTOYBACNPR1,1,49,0.000001,2.018219,TOYB3,TOYB4,Consumo cíclico,Consumo cíclico,True,Consumo cíclico | Consumo cíclico


### Criação dos sinais do modelo

Nesta etapa, criamos sinais com base no z-score dinâmico calculado pelo Kalman.

A regra usada será:

- se z-score >= 2: o ativo 1 está caro em relação ao ativo 2;
- se z-score <= -2: o ativo 1 está barato em relação ao ativo 2;
- caso contrário: sem sinal.

Esses sinais indicam possíveis momentos de entrada.

In [14]:
modelo_kalman["sinal"] = 0

modelo_kalman.loc[
    modelo_kalman["zscore_kalman"] >= Z_ENTRADA,
    "sinal"
] = -1

modelo_kalman.loc[
    modelo_kalman["zscore_kalman"] <= -Z_ENTRADA,
    "sinal"
] = 1

modelo_kalman["direcao"] = "NEUTRO"

modelo_kalman.loc[
    modelo_kalman["sinal"] == -1,
    "direcao"
] = "SHORT_ATIVO_1_LONG_ATIVO_2"

modelo_kalman.loc[
    modelo_kalman["sinal"] == 1,
    "direcao"
] = "LONG_ATIVO_1_SHORT_ATIVO_2"

display(modelo_kalman.head(20))

,data,alpha_kalman,beta_kalman,spread_kalman,variancia_inovacao,zscore_kalman,data_fim_janela,data_fim_teste,ativo_1,ativo_2,ranking_cointegracao,ranking_distancia,pvalor_coint,distancia,ticker_1,ticker_2,setor_1,setor_2,mesmo_setor,combinacao_setorial,sinal,direcao
0,2012-02-01,0.192486,0.927505,0.017306,13.309462,0.004744,2012-01-31,2012-02-29,BRTOYBACNOR4,BRTOYBACNPR1,1,49,0.000001,2.018219,TOYB3,TOYB4,Consumo cíclico,Consumo cíclico,True,Consumo cíclico | Consumo cíclico,0,NEUTRO
1,2012-02-02,0.160653,0.934558,-0.016159,0.030383,-0.092702,2012-01-31,2012-02-29,BRTOYBACNOR4,BRTOYBACNPR1,1,49,0.000001,2.018219,TOYB3,TOYB4,Consumo cíclico,Consumo cíclico,True,Consumo cíclico | Consumo cíclico,0,NEUTRO
2,2012-02-03,0.152677,0.937595,0.007120,0.022523,0.047443,2012-01-31,2012-02-29,BRTOYBACNOR4,BRTOYBACNPR1,1,49,0.000001,2.018219,TOYB3,TOYB4,Consumo cíclico,Consumo cíclico,True,Consumo cíclico | Consumo cíclico,0,NEUTRO
3,2012-02-06,0.522811,0.844369,0.213666,0.020093,1.507353,2012-01-31,2012-02-29,BRTOYBACNOR4,BRTOYBACNPR1,1,49,0.000001,2.018219,TOYB3,TOYB4,Consumo cíclico,Consumo cíclico,True,Consumo cíclico | Consumo cíclico,0,NEUTRO
4,2012-02-07,0.425407,0.868761,-0.073522,0.018405,-0.541934,2012-01-31,2012-02-29,BRTOYBACNOR4,BRTOYBACNPR1,1,49,0.000001,2.018219,TOYB3,TOYB4,Consumo cíclico,Consumo cíclico,True,Consumo cíclico | Consumo cíclico,0,NEUTRO
5,2012-02-08,0.464392,0.855821,-0.026920,0.018446,-0.198204,2012-01-31,2012-02-29,BRTOYBACNOR4,BRTOYBACNPR1,1,49,0.000001,2.018219,TOYB3,TOYB4,Consumo cíclico,Consumo cíclico,True,Consumo cíclico | Consumo cíclico,0,NEUTRO
6,2012-02-09,0.747319,0.761383,-0.243677,0.017541,-1.839866,2012-01-31,2012-02-29,BRTOYBACNOR4,BRTOYBACNPR1,1,49,0.000001,2.018219,TOYB3,TOYB4,Consumo cíclico,Consumo cíclico,True,Consumo cíclico | Consumo cíclico,0,NEUTRO
7,2012-02-10,0.720712,0.770327,0.027684,0.016987,0.212405,2012-01-31,2012-02-29,BRTOYBACNOR4,BRTOYBACNPR1,1,49,0.000001,2.018219,TOYB3,TOYB4,Consumo cíclico,Consumo cíclico,True,Consumo cíclico | Consumo cíclico,0,NEUTRO
8,2012-02-13,1.016484,0.690921,0.194823,0.017158,1.487350,2012-01-31,2012-02-29,BRTOYBACNOR4,BRTOYBACNPR1,1,49,0.000001,2.018219,TOYB3,TOYB4,Consumo cíclico,Consumo cíclico,True,Consumo cíclico | Consumo cíclico,0,NEUTRO
9,2012-02-14,1.223624,0.622340,-0.217552,0.016616,-1.687701,2012-01-31,2012-02-29,BRTOYBACNOR4,BRTOYBACNPR1,1,49,0.000001,2.018219,TOYB3,TOYB4,Consumo cíclico,Consumo cíclico,True,Consumo cíclico | Consumo cíclico,0,NEUTRO


### Checagem dos sinais gerados

Nesta etapa, verificamos quantos sinais foram gerados pelo modelo.

Isso permite entender se o modelo está gerando sinais demais, sinais de menos ou se a maioria dos períodos está neutra.

In [15]:
resumo_sinais = (
    modelo_kalman
    .groupby("direcao")
    .size()
    .reset_index(name="qtd_observacoes")
    .sort_values("qtd_observacoes", ascending=False)
)

display(resumo_sinais)

,direcao,qtd_observacoes
1,NEUTRO,67009
2,SHORT_ATIVO_1_LONG_ATIVO_2,249
0,LONG_ATIVO_1_SHORT_ATIVO_2,188


### Checagem de sinais por janela

Nesta etapa, verificamos quantos sinais foram gerados em cada janela de formação.

In [17]:
sinais_por_janela = (
    modelo_kalman
    .assign(tem_sinal=modelo_kalman["sinal"] != 0)
    .groupby("data_fim_janela")["tem_sinal"]
    .sum()
    .reset_index(name="qtd_sinais")
)

display(sinais_por_janela.describe())

display(sinais_por_janela.head())

display(sinais_por_janela.tail())

,data_fim_janela,qtd_sinais
count,172,172.000000
mean,2019-03-16 13:06:58.604651,2.540698
min,2012-01-31 00:00:00,0.000000
25%,2015-08-23 06:00:00,0.000000
50%,2019-03-15 12:00:00,1.000000
75%,2022-10-07 18:00:00,4.000000
max,2026-04-30 00:00:00,31.000000
std,NaN,4.120066


,data_fim_janela,qtd_sinais
0,2012-01-31,0
1,2012-02-29,1
2,2012-03-31,2
3,2012-04-30,1
4,2012-05-31,4


,data_fim_janela,qtd_sinais
167,2025-12-31,2
168,2026-01-31,6
169,2026-02-28,3
170,2026-03-31,0
171,2026-04-30,0


### Checagem do beta dinâmico

Nesta etapa, analisamos o beta estimado pelo Kalman.

O beta dinâmico é importante porque representa o hedge ratio do par ao longo do tempo.

Se o beta varia muito, isso indica que a relação entre os ativos é instável.

In [18]:
resumo_beta = (
    modelo_kalman
    .groupby(["data_fim_janela", "ativo_1", "ativo_2"])
    .agg(
        beta_medio=("beta_kalman", "mean"),
        beta_final=("beta_kalman", "last"),
        beta_desvio=("beta_kalman", "std"),
        zscore_final=("zscore_kalman", "last")
    )
    .reset_index()
)

display(resumo_beta.head())

display(resumo_beta.describe())

,data_fim_janela,ativo_1,ativo_2,beta_medio,beta_final,beta_desvio,zscore_final
0,2012-01-31,BRABCBACNPR4,BRITSAACNPR7,1.119741,0.968660,0.153165,0.580227
1,2012-01-31,BRABCBACNPR4,BRITUBACNPR1,1.166851,1.078211,0.079111,0.426444
2,2012-01-31,BRABEVACNOR1,BRAMBVACNPR1,0.986008,1.088659,0.060923,0.333724
3,2012-01-31,BRAGROACNOR7,BRSLCEACNOR2,0.385131,0.376247,0.062067,0.013711
4,2012-01-31,BRB3SAACNOR6,BROGXPACNOR3,0.544277,0.529842,0.007839,-0.113415


,data_fim_janela,beta_medio,beta_final,beta_desvio,zscore_final
count,3383,"3,383.000000","3,383.000000","3,383.000000","3,383.000000"
mean,2019-03-06 00:44:16.104049,0.753179,0.674336,0.102934,0.041900
min,2012-01-31 00:00:00,-0.374911,-1.366218,0.000877,-4.374397
25%,2015-07-31 00:00:00,0.553221,0.429838,0.036298,-0.314409
50%,2019-02-28 00:00:00,0.786271,0.719964,0.073153,0.024967
75%,2022-09-30 00:00:00,0.959251,0.929073,0.136091,0.375679
max,2026-04-30 00:00:00,2.042378,3.776573,1.318891,8.164590
std,NaN,0.305079,0.378906,0.101499,0.662976


### Salvamento da base do modelo

Por fim, salvamos a base com os resultados do Filtro de Kalman.

Essa base será usada no próximo notebook para realizar o backtest.

A base contém:

- alpha dinâmico;
- beta dinâmico;
- spread dinâmico;
- z-score dinâmico;
- sinais de trading;
- informações dos pares e setores.

In [19]:
modelo_kalman.to_parquet(arquivo_saida, index=False)

print("Base do modelo Kalman salva em:", arquivo_saida)

Base do modelo Kalman salva em: ..\dados_tratados\modelo_kalman_sinais.parquet


### Conferência do arquivo salvo

In [20]:
modelo_salvo = pd.read_parquet(arquivo_saida)

print("Base salva:", modelo_salvo.shape)

display(modelo_salvo.head())

Base salva: (67446, 22)


,data,alpha_kalman,beta_kalman,spread_kalman,variancia_inovacao,zscore_kalman,data_fim_janela,data_fim_teste,ativo_1,ativo_2,ranking_cointegracao,ranking_distancia,pvalor_coint,distancia,ticker_1,ticker_2,setor_1,setor_2,mesmo_setor,combinacao_setorial,sinal,direcao
0,2012-02-01,0.192486,0.927505,0.017306,13.309462,0.004744,2012-01-31,2012-02-29,BRTOYBACNOR4,BRTOYBACNPR1,1,49,0.000001,2.018219,TOYB3,TOYB4,Consumo cíclico,Consumo cíclico,True,Consumo cíclico | Consumo cíclico,0,NEUTRO
1,2012-02-02,0.160653,0.934558,-0.016159,0.030383,-0.092702,2012-01-31,2012-02-29,BRTOYBACNOR4,BRTOYBACNPR1,1,49,0.000001,2.018219,TOYB3,TOYB4,Consumo cíclico,Consumo cíclico,True,Consumo cíclico | Consumo cíclico,0,NEUTRO
2,2012-02-03,0.152677,0.937595,0.007120,0.022523,0.047443,2012-01-31,2012-02-29,BRTOYBACNOR4,BRTOYBACNPR1,1,49,0.000001,2.018219,TOYB3,TOYB4,Consumo cíclico,Consumo cíclico,True,Consumo cíclico | Consumo cíclico,0,NEUTRO
3,2012-02-06,0.522811,0.844369,0.213666,0.020093,1.507353,2012-01-31,2012-02-29,BRTOYBACNOR4,BRTOYBACNPR1,1,49,0.000001,2.018219,TOYB3,TOYB4,Consumo cíclico,Consumo cíclico,True,Consumo cíclico | Consumo cíclico,0,NEUTRO
4,2012-02-07,0.425407,0.868761,-0.073522,0.018405,-0.541934,2012-01-31,2012-02-29,BRTOYBACNOR4,BRTOYBACNPR1,1,49,0.000001,2.018219,TOYB3,TOYB4,Consumo cíclico,Consumo cíclico,True,Consumo cíclico | Consumo cíclico,0,NEUTRO
